# Lab 7.4 &mdash; Extract: Mess In, Structure Out

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 25 min &nbsp;|&nbsp; **Day 4 &middot; Module 7 &mdash; Task Automation with AI Agents**

### What you'll do
- Ask the real model for structured output that fills a tight schema
- Handle a missing order id (null, not invented)
- Validate the model's intent against a closed set before trusting it

> **How this lab works (near-real):** you have a `GROQ_API_KEY` in the repo `.env`. Read the **Concept**, fill the real `___` blanks in **Build it** (real pipeline logic, real tool bodies, the real draft/`create_agent` call), then **Run it for real** &mdash; a real Groq model drives the step over real tools &mdash; and **read the output/trace**. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a working email agent and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langchain-groq`) and a **real hosted model** (`ChatGroq("openai/gpt-oss-20b")` &mdash; reliable tool-calling via `create_agent`). If `GROQ_API_KEY` is unset, the run cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. You are building the **email-drafting agent** (the client's Lab 4.1): it **drafts but never sends** &mdash; `send_email` is withheld and a human approves.

**Reference:** [Module 7 slides &mdash; Extract — mess in, structure out](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 7 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, time, socket, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (+ other keys)

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-07-04")
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is configured -- the 'Run it for real' cells self-skip if not."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-4 provider: a REAL hosted model with reliable tool-calling via create_agent.
MODEL = "openai/gpt-oss-20b"
llm = ChatGroq(model=MODEL, temperature=0) if groq_ready() else None

def with_backoff(fn, tries=4):
    """Run fn(); retry with backoff on Groq rate limits (HTTP 429). Other errors propagate."""
    last = None
    for attempt in range(tries):
        try:
            return fn()
        except Exception as e:
            last = e
            if "429" in str(e) or "rate limit" in str(e).lower() or "rate_limit" in str(e).lower():
                wait = 5 * (attempt + 1)
                print(f"(rate-limited -- retrying in {wait}s)")
                time.sleep(wait)
                continue
            raise
    raise last

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("Groq ready | model:", MODEL, "| WORK:", WORK)
else:
    print("GROQ_API_KEY not set -- add it to the repo .env (free key at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
**Extract** turns unstructured input into structured data (deck slide 10): an email *"my order from
last Tuesday still hasn't arrived, ref 4471, getting frustrated"* becomes
`{"order_id": "4471", "intent": "delivery", "sentiment": "negative"}`. The reliable way is the **extract
pattern** &mdash; ask the **real model** for **structured output** (`llm.with_structured_output(schema)`),
so the model reads the language and fills a **tight schema** you define (only the fields you'll use).
But you **never trust it blindly**: constrain `intent` to a **closed set** (with an escape hatch) so a
stray label can't break the router. Extract is the **first step** in the chain &mdash; extract &rarr;
route &rarr; draft.

In [ ]:
sample = "Hi, my order from last Tuesday still hasn't arrived, ref 4471, getting frustrated."
print("unstructured in:", sample)
print("we want out    : {order_id, intent, sentiment} -- filled by the real model, guarded by a closed set")

## Build it
Complete `extract`: bind the `Extracted` schema to the model with `with_structured_output`, then constrain
`intent` to the closed set.

In [ ]:
from typing import Optional
from pydantic import BaseModel, Field

INTENTS = ("refund", "delivery", "cancel", "other")

class Extracted(BaseModel):
    """The tight schema we pull from a messy customer email."""
    order_id: Optional[str] = Field(description="the order number as a string, or null if none")
    intent: str = Field(description="one of: refund, delivery, cancel, other")
    sentiment: str = Field(description="negative or neutral")

def extract(email):
    """Use the REAL model as an EXTRACTOR (structured output), then validate against the closed set."""
    structured = ___   # TODO: make llm return the Extracted schema (hint: llm.with_structured_output(Extracted, method="json_schema"))
    result = with_backoff(lambda: structured.invoke(
        "Extract the fields from this customer email. Use ONLY the email text; "
        "do not call any tool or search the web.\nEmail: " + email))
    rec = result.model_dump()
    # never trust the model blindly: constrain intent to the CLOSED set (the escape hatch)
    if ___:   # TODO: the model returned an intent OUTSIDE the closed INTENTS set
        rec["intent"] = "other"
    return rec

## Run it for real &amp; read the trace
Run the real extractor over a few emails &mdash; including *'my parcel never showed up'*, which has no delivery keyword. The model reads meaning, not keywords.

_This calls the real `openai/gpt-oss-20b` via Groq. If `GROQ_API_KEY` is unset the cell prints how to set it instead of crashing._

In [ ]:
if not groq_ready():
    print("No GROQ_API_KEY -- add it to the repo .env (free key at console.groq.com), then re-run this cell.")
else:
    try:
        for email in ["my order 4471 still hasn't arrived, getting frustrated",
                      "my parcel never showed up",          # NO delivery keyword -- the model still gets it
                      "please cancel my subscription",
                      "I want a refund"]:
            print(repr(email[:40]), "->", extract(email))
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- The model fills the **schema** directly &mdash; `with_structured_output(..., method="json_schema")` makes Groq return JSON matching `Extracted`, so the reply is a typed object, not prose you must parse (and `json_schema` mode is more robust than the tool-calling default on this model).
- *'my parcel never showed up'* has **no delivery keyword**, yet the model labels it `delivery` &mdash; exactly the phrasing a keyword extractor missed. The model reads meaning.
- You still **validate**: an intent outside the closed set is forced to `other`, so a stray label can never break the router. Trust, but verify.

## Your turn (open task &mdash; no grader)
Feed `extract` your own awkward emails &mdash; sarcasm, two intents in one message, a typo'd order number &mdash;
and see where the model's structured output surprises you. **What good looks like:** the schema is always
well-formed and `intent` is always in the closed set &mdash; the model does the reading, your validator keeps it
safe.

---
### Key takeaway
The extract pattern uses the real model's language understanding to fill a tight schema -- and a closed-set guard keeps its output safe to branch on. That's the workhorse first step of every automation.

[&#8592; All Module 7 labs](./index.html) &nbsp;&middot;&nbsp; [Module 7 slides](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>